# Kaggle Competition info:
## NVIDIA Nemotron Model Reasoning Challenge
#### Advance reasoning techniques using NVIDIA Nemotron open models on a novel benchmark

## Overview
Develop techniques that improve reasoning accuracy using NVIDIA Nemotron models.

Participants will experiment with prompting, data pipelines, and lightweight fine-tuning while evaluating their approaches on a new reasoning benchmark developed by NVIDIA Research.

**Link:** <https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/overview>

## Install libary

In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os


In [8]:
# from trl import DPOTrainer, DPOConfig
# from peft import PeftModel, LoraConfig
# from datasets import load_dataset
# import torch

import os, glob, sys, subprocess, site, importlib.util, shutil, stat, types, re

import datasets
import kagglehub

import torch
# import mamba_ssm later

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

import pandas as pd
import random
import gc, time

import json, zipfile

In [9]:
class CFG:
    trainFile = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
    testFile = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv"

    llmModel1 = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

In [11]:
trainDF = pd.read_csv(CFG.trainFile)
trainDF

,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret
...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates


In [12]:
testDF = pd.read_csv(CFG.testFile)
testDF

,id,prompt
0,00066667,"In Alice's Wonderland, a secret bit manipulati..."
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati..."
2,00189f6a,"In Alice's Wonderland, secret encryption rules..."


In [14]:
print(trainDF["prompt"][0])

In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100
